In [2]:
!pip install mne -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [3]:
import mne
import numpy as np

In [4]:
from mne.datasets import eegbci

In [ ]:
# Connect to Google drive

from google.colab import drive
import os

drive.mount("/content/drive", force_remount = True)


PROJECT_DIR = "/content/drive/MyDrive/neurosynth"

os.makedirs(PROJECT_DIR, exist_ok = True)

In [ ]:
# Loading the dataset

DATA_DIR = "/content/drive/MyDrive/neurosynth/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/neurosynth/data/processed"



SUBJECTS = [1, 2, 3, 4, 5]
RUNS = [4, 8, 12]


# Bandpass filter - keeps only Alpha + Beta motor imagery frequencies
# validated earlier with our PSD plot
FREQ_LOW = 8.0
FREQ_HIGH = 30.0

# Epoch window - 4 second task windows matching experiment design
EPOCH_TMIN = 0.0
EPOCH_TMAX = 4.0

# T1 - left fist imagery --> label 0
# T2 - right fist imagery ----> label 1
EVENT_ID = {"T1": 1, "T2": 2}


os.makedirs(OUTPUT_DIR, exist_ok = True)
print("Configuration set!")


In [ ]:
# see exactly what files exist for one subject
from mne.datasets import eegbci


# request ALL 14 runs to see the full picture
all_runs = list(range(1,15)) # 1 to 14

all_fnames = eegbci.load_data(
    1,
    runs = all_runs,
    path = DATA_DIR,
    verbose = False
)


print("All 14 runs for Subject 1")
for i, fname in enumerate(all_fnames, start = 1):
  print(f"Run {i:2d}: {fname}")

In [ ]:
import json
# shutil = copy files between folders
import shutil
import os
import subprocess

# datetime = get current date/time for commit messages
from datetime import datetime
from google.colab import userdata


GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "the-liyanage"
REPO_NAME = "neurosynth"


# 1. set git identity
subprocess.run(["git", "config", "--global",
               "user.name", "the-liyanage"])

subprocess.run(["git", "config", "--global",
                "user.email", "hiruniliyanage4@gmail.com"])


# 2. Clone repo fresh (only if not already there)
if not os.path.exists(f"/content/{REPO_NAME}/.git"):
  subprocess.run(["rm", "-rf", "f/content/{REPO_NAME}"])
  os.chdir("/content")
  subprocess.run([
      "git", "clone",
      f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
      ])
  print("Repo clones fresh!")
else:
  print("Repo already exisit, skipping clone")




# 3. Copy notebook into repo
os.makedirs(f"/content/{REPO_NAME}/notebooks", exist_ok = True)
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/02_preprocessing.ipynb",
    f"/content/neurosynth/notebooks/02_preprocessing.ipynb"
)
print(f"\n Notebook copied!")


# 4. Commit and push

# move into the repo folder
os.chdir(f"/content/{REPO_NAME}")

# stage all change
subprocess.run(["git", "add", "."])


commit_message = "set configurations"

subprocess.run(["git", "commit", "-m", commit_message])


result = subprocess.run([
    "git", "push",
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git",
    "main"
], capture_output=True, text=True)


if result.returncode == 0:
  print("Pushed to the github", commit_message)
else:
  print("Push failed")
  print(result.stderr)